# GLM 5.2 — Cookbook

Practical recipes for building with GLM 5.2 through OpenRouter's OpenAI-compatible API.

> Source: Z.AI GLM-5.2 docs — https://docs.z.ai/guides/llm/glm-5.2
> 
> Recommended OpenRouter model id: `z-ai/glm-5.2`

## 0) What GLM 5.2 is good at

GLM 5.2 is designed for long-horizon engineering work. The docs highlight:
- 1M-token context for project-scale tasks
- Stronger coding and refactoring stability over long tasks
- Text input and text output
- Support for MCP-style tool integration in agentic workflows

In practice, this cookbook focuses on:
- coding and refactoring
- long-context summarization
- multi-step reasoning
- structured outputs
- OpenRouter-based calling patterns

## 1) Setup

Install the OpenAI Python SDK, then point it at OpenRouter.

- `pip install openai`
- Set `OPENROUTER_API_KEY`
- Optional but recommended: set `OPENROUTER_HTTP_REFERER` and `OPENROUTER_APP_TITLE` for attribution and routing

In [ ]:
import json
import os
from pathlib import Path

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None
    print('Install the OpenAI SDK with: %pip install openai')

api_key = os.getenv('OPENROUTER_API_KEY')
if OpenAI is None:
    client = None
elif api_key:
    client = OpenAI(
        base_url='https://openrouter.ai/api/v1',
        api_key=api_key,
    )
    print('OpenRouter client configured')
else:
    client = None
    print('Set OPENROUTER_API_KEY to run live requests')

## 2) Basic OpenRouter chat request

OpenRouter is OpenAI-compatible, so the basic structure is a standard chat/completions call. Use the GLM 5.2 model id and a system prompt that clearly assigns the role.

In [ ]:
payload = {
    'model': 'z-ai/glm-5.2',
    'messages': [
        {
            'role': 'system',
            'content': 'You are a senior engineering assistant. Be precise, structured, and resilient on long tasks.'
        },
        {
            'role': 'user',
            'content': 'Design a clean architecture for a multi-tenant document workflow app and list the key modules.'
        }
    ],
    'temperature': 0.2,
    'max_tokens': 2048
}

print(json.dumps(payload, indent=2))

## 3) Long-horizon coding recipe

GLM 5.2 is positioned for long projects, so prompt it with a concrete deliverable, a task boundary, and a clear stopping condition.

Good pattern:
- give the repo context
- define the exact output artifact
- require a short plan before edits
- ask for a validation step

In [ ]:
coding_prompt = '''
You are working in a large codebase.
Task: add a retry-safe background worker for failed document sync jobs.
Constraints:
- preserve existing APIs
- add tests for the new worker
- do not change database schema unless necessary
Output:
1. plan
2. files to edit
3. patch summary
4. validation steps
'''

print(coding_prompt)

## 4) Summarization and structured output

For 1M-context workflows, chunk large inputs and ask the model to produce a compact summary for each chunk before a final consolidation pass.

For structured output, ask for JSON and include an example schema.

In [ ]:
chunk_summarize = {
    'system': 'Summarize each chunk faithfully, preserving names, dates, and code symbols.',
    'user': 'Chunk 1 of 4: [paste text here]'
}

structured_output_prompt = {
    'system': 'Return valid JSON only.',
    'user': 'Extract the action items from this plan and return {"items": [{"title": str, "owner": str, "due": str}]}'
}

print(json.dumps(chunk_summarize, indent=2))
print(json.dumps(structured_output_prompt, indent=2))

## 5) Minimal validation and next steps

A practical cookbook should include a quick check that the notebook loads and contains the expected sections. If you want, I can also add a runnable HTTP example cell that uses `requests` instead of the OpenAI SDK.

## 2.1) Direct HTTP example with `requests`

If you prefer not to use the SDK, OpenRouter also works with a plain POST request to the OpenAI-compatible chat completions endpoint.

In [ ]:
import requests

http_example = {
    'url': 'https://openrouter.ai/api/v1/chat/completions',
    'headers': {
        'Authorization': 'Bearer $OPENROUTER_API_KEY',
        'Content-Type': 'application/json',
        'HTTP-Referer': os.getenv('OPENROUTER_HTTP_REFERER', 'https://example.com'),
        'X-Title': os.getenv('OPENROUTER_APP_TITLE', 'GLM 5.2 Cookbook'),
    },
    'json': {
        'model': 'z-ai/glm-5.2',
        'messages': [
            {'role': 'system', 'content': 'You are a senior engineer who thinks carefully about long-horizon tasks.'},
            {'role': 'user', 'content': 'Summarize this 30-page design doc into implementation milestones.'},
        ],
        'temperature': 0.2,
        'max_tokens': 2048,
    },
}

print(json.dumps(http_example, indent=2))

## 6) LangChain integration

Use LangChain when you want prompt templates, retrievers, structured outputs, and composable chains. This is especially useful for GLM 5.2 because the model is positioned for long-horizon workflows where you want to split planning, retrieval, and generation into separate steps.

### 6.1) Install the LangChain stack

If the environment does not already have LangChain, install the packages below. `langchain-openai` gives you the OpenAI-compatible client wrapper that works with OpenRouter.

In [ ]:
# Run once if needed
# %pip install -U langchain langchain-openai langchain-core langchain-text-splitters tiktoken

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

lc_model = ChatOpenAI(
    model='z-ai/glm-5.2',
    base_url='https://openrouter.ai/api/v1',
    api_key=os.getenv('OPENROUTER_API_KEY'),
    default_headers={
        'HTTP-Referer': os.getenv('OPENROUTER_HTTP_REFERER', 'https://example.com'),
        'X-Title': os.getenv('OPENROUTER_APP_TITLE', 'GLM 5.2 Cookbook'),
    },
    temperature=0.2,
)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a senior software architect. Prefer concise but actionable answers.'),
    ('user', 'Given this feature request, produce a one-paragraph plan and five implementation bullets: {request}'),
])

chain = prompt | lc_model | StrOutputParser()
print('LangChain chain constructed for OpenRouter + GLM 5.2')

## 7) Long-context summarization with chunking

A strong GLM 5.2 pattern is to process a large document in chunks, summarize each chunk, then combine the summaries into a final answer. This keeps the model grounded and makes the pipeline easier to debug.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=2500, chunk_overlap=250)
chunks = text_splitter.split_text(
    'Paste a long spec, design doc, or repository readme here before summarizing it with GLM 5.2.'
)
print('Chunk count:', len(chunks))
print('First chunk preview:', chunks[0][:300] if chunks else 'no chunks')

## 8) Structured outputs and tool use

GLM 5.2 is a good fit for workflows where you want a strict output schema or a tool-assisted reasoning loop. Use structured outputs for plans, issue triage, and agent handoffs. Use tool calling when you want the model to inspect or transform external data.

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class ActionPlan(BaseModel):
    summary: str = Field(..., description='One-paragraph summary of the task')
    milestones: List[str] = Field(..., description='Ordered implementation milestones')
    risks: List[str] = Field(..., description='Key technical risks or dependencies')

structured_llm = lc_model.with_structured_output(ActionPlan)
print('Structured output wrapper ready (schema-based planning)')

## 9) Evaluation and model-routing ideas

A more advanced cookbook benefits from a repeatable evaluation harness and a routing strategy:
- use GLM 5.2 for deep reasoning and project-scale tasks
- use a faster model for cheap drafts or classification
- rerun the result through GLM 5.2 as a verifier or refiner
- record prompt/output pairs for regression testing

In [ ]:
def route_task(task_kind: str) -> str:
    if task_kind in {'deep_reasoning', 'code_review', 'long_context'}:
        return 'z-ai/glm-5.2'
    if task_kind in {'draft', 'classification'}:
        return 'fast_openrouter_model'
    return 'z-ai/glm-5.2'

for task_kind in ['deep_reasoning', 'draft', 'long_context']:
    print(task_kind, '->', route_task(task_kind))